In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Reviews.csv")
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [3]:
df = df.sample(20000, random_state=42)

In [4]:
df = df[['Text', 'Score']]
df.head()

,Text,Score
165256,Having tried a couple of other brands of glute...,5
231465,My cat loves these treats. If ever I can't fin...,5
427827,A little less than I expected. It tends to ha...,3
433954,"First there was Frosted Mini-Wheats, in origin...",2
70260,and I want to congratulate the graphic artist ...,5


In [5]:
def convert_sentiment(score):
    if score >= 4:
        return 1   # Positive
    elif score <= 2:
        return 0   # Negative
    else:
        return None  # Neutral

df['sentiment'] = df['Score'].apply(convert_sentiment)

In [6]:
df = df.dropna()

In [7]:
df['sentiment'].value_counts()

sentiment
1.0    15595
0.0     2886
Name: count, dtype: int64

In [12]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [10]:
def preprocess_text(text):
   
    text = text.lower()
    
    text = re.sub(r"http\S+|www\S+", "", text)
    
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    tokens = word_tokenize(text)
    
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

In [13]:
df['clean_text'] = df['Text'].apply(preprocess_text)

In [14]:
df[['Text', 'clean_text']].head()

,Text,clean_text
165256,Having tried a couple of other brands of glute...,tri coupl brand gluten free sandwich cooki bes...
231465,My cat loves these treats. If ever I can't fin...,cat love treat ever find hous pop top bolt whe...
433954,"First there was Frosted Mini-Wheats, in origin...",first frost mini wheat origin size frost mini ...
70260,and I want to congratulate the graphic artist ...,want congratul graphic artist put entir produc...
49866,Please add more Pineapple flavor to your packa...,pleas add pineappl flavor packag lifesav fact ...


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_text'])
y = df['sentiment']

In [16]:
print(X.shape)

(18481, 5000)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.9034352177441168
              precision    recall  f1-score   support

         0.0       0.86      0.44      0.59       569
         1.0       0.91      0.99      0.95      3128

    accuracy                           0.90      3697
   macro avg       0.88      0.72      0.77      3697
weighted avg       0.90      0.90      0.89      3697



In [19]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.8574519880984582
              precision    recall  f1-score   support

         0.0       0.89      0.08      0.15       569
         1.0       0.86      1.00      0.92      3128

    accuracy                           0.86      3697
   macro avg       0.87      0.54      0.54      3697
weighted avg       0.86      0.86      0.80      3697



In [20]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

Decision Tree Accuracy: 0.8498782796862321
              precision    recall  f1-score   support

         0.0       0.51      0.50      0.51       569
         1.0       0.91      0.91      0.91      3128

    accuracy                           0.85      3697
   macro avg       0.71      0.71      0.71      3697
weighted avg       0.85      0.85      0.85      3697



In [21]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        0.9034,
        0.8574,
        0.8498
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.9034
1,Naive Bayes,0.8574
2,Decision Tree,0.8498


## Conclusion

In this project we built a complete sentiment analysis system using the Amazon Fine Food Reviews dataset

We applied NLP preprocessing TF-IDF feature extraction, and trained multiple machine learning models.

Among all models, Logistic Regression performed best with the highest accuracy.

This project demonstrates how NLP and machine learning can be used for real-world sentiment classification tasks.